<a href="https://colab.research.google.com/github/manasah3/CS598-DLH/blob/main/colab_dreamtdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 598 DL4H — WatchSleepNet Colab POC

This notebook validates the Colab ↔ GitHub ↔ Google Drive workflow for the WatchSleepNet replication project.

**Branch:** `colab-poc`  
**Repo:** [cs598-DLH-WatchSleepNet](https://github.com/jananij2/cs598-DLH-WatchSleepNet)

## 1. Install Dependencies

In [1]:
# Install PyHealth from source (use this during active contribution work)
!pip install git+https://github.com/sunlabuiuc/PyHealth.git -q

# Other common dependencies
!pip install numpy pandas scikit-learn matplotlib -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import pyhealth
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyHealth version: {pyhealth.__version__}")
print("All imports successful.")

PyHealth version: 2.0.0
All imports successful.


## 2. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

# Base path — adjust if your Drive folder structure differs
DRIVE_BASE = '/content/drive/MyDrive/cs598-watchsleepnet'

PATHS = {
    'raw':       os.path.join(DRIVE_BASE, 'raw'),
    'processed': os.path.join(DRIVE_BASE, 'processed'),
    'models':    os.path.join(DRIVE_BASE, 'models'),
    'results':   os.path.join(DRIVE_BASE, 'results'),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"{name:12s} → {path}")

raw          → /content/drive/MyDrive/cs598-watchsleepnet/raw
processed    → /content/drive/MyDrive/cs598-watchsleepnet/processed
models       → /content/drive/MyDrive/cs598-watchsleepnet/models
results      → /content/drive/MyDrive/cs598-watchsleepnet/results


## 3. Dataset Implementation (dreamt.py)


In [8]:
import os
import pandas as pd
from typing import Dict, List, Any, Optional
from pyhealth.datasets import BaseDataset

class DREAMTDataset(BaseDataset):
    """Dataset for Real-time sleep stage Estimation (DREAMT)."""

    def __init__(self, root: str, dev: bool = True, **kwargs):
        import time
        # We use a unique name to ensure we don't load a broken old cache
        unique_name = f"DREAMT_{int(time.time())}"
        super(DREAMTDataset, self).__init__(
            dataset_name=unique_name,
            root=root,
            tables=["whole_df"],
            dev=dev,
            **kwargs
        )

    def parse_basic_info(self) -> Dict[str, Dict[str, Any]]:
        """Parses demographics from participant_info.csv."""
        info_path = os.path.join(self.root, "participant_info.csv")
        df = pd.read_csv(info_path)

        # Standardize column names to uppercase to match your file
        df.columns = df.columns.str.strip().str.upper()
        print(f"DEBUG: Successfully mapped headers: {df.columns.tolist()}")

        data = {}
        for _, row in df.iterrows():
            # Your SID already looks like 'S002', so we use it directly
            sid = str(row["SID"]).strip()
            data[sid] = {
                "age": row.get("AGE", 0),
                "gender": row.get("GENDER", "Unknown"),
                "ahi": row.get("AHI", 0)
            }
        return data

    def parse_tables(self) -> Dict[str, Dict[str, List]]:
        """Parses the time-series signal CSVs."""
        patients = self.parse_basic_info()
        data_dir = os.path.join(self.root, "data_64Hz")

        final_patients = {}

        for sid in list(patients.keys()):
            file_path = os.path.join(data_dir, f"{sid}_whole_df.csv")

            if os.path.exists(file_path):
                # Load the 133MB file
                df = pd.read_csv(file_path)

                # Standardize signal columns
                df.columns = df.columns.str.strip()

                final_patients[sid] = {
                    "patient_id": sid,
                    "age": patients[sid]["age"],
                    "gender": patients[sid]["gender"],
                    # Map exactly to the columns in S00x_whole_df.csv
                    "ibi": df["IBI"].tolist(),
                    "label": df["Sleep_Stage"].tolist()
                }

        self.patients = final_patients
        return final_patients

## 4. Task Implementation (sleep_staging_dreamt.py)

In [9]:
from typing import Dict, List

def sleep_staging_dreamt_fn(patient: Dict) -> List[Dict]:
    """Task function to window IBI data into 30-second epochs.

    Window Logic: 30 seconds @ 64Hz = 1920 samples.
    """
    samples = []
    # Note: We use the keys we defined in the Dataset class
    ibi = patient.get("ibi", [])
    labels = patient.get("label", [])

    window_size = 1920
    label_map = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}

    for i in range(0, len(ibi) - window_size, window_size):
        epoch_ibi = ibi[i : i + window_size]
        epoch_label = labels[i + window_size - 1]

        if epoch_label in label_map:
            samples.append({
                "patient_id": patient["patient_id"],
                "feature": epoch_ibi,
                "label": label_map[epoch_label]
            })
    return samples

# REQUIRED FOR PYHEALTH RUBRIC:
sleep_staging_dreamt_fn.task_name = "sleep_staging_dreamt"

## 5.  Execution and Verification


In [10]:
print("--- Step 1: Loading Dataset ---")
dataset = DREAMTDataset(root=PATHS['raw'], dev=True)

# If PyHealth's internal loading failed, we force our parsed data in
if not hasattr(dataset, 'patients') or len(dataset.patients) == 0:
    print("Forcing manual parse...")
    dataset.patients = dataset.parse_tables()

print(f"✅ Success! {len(dataset.patients)} subjects ready.")

# 2. Apply Task
print("\n--- Step 2: Applying Task ---")
# We use the dataset's own logic to process the patients
samples = []
for sid, patient in dataset.patients.items():
    samples.extend(sleep_staging_dreamt_fn(patient))

print(f"✅ Total sleep epochs created: {len(samples)}")

if len(samples) > 0:
    print(f"Sample 0 Feature Length: {len(samples[0]['feature'])} (Expected: 1920)")
    print(f"Sample 0 Label: {samples[0]['label']}")

--- Step 1: Loading Dataset ---
Initializing DREAMT_1775099969 dataset from /content/drive/MyDrive/cs598-watchsleepnet/raw (dev mode: True)


INFO:pyhealth.datasets.base_dataset:Initializing DREAMT_1775099969 dataset from /content/drive/MyDrive/cs598-watchsleepnet/raw (dev mode: True)


No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/d6a1135d-6371-5b03-9ab0-690b651231ab


INFO:pyhealth.datasets.base_dataset:No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/d6a1135d-6371-5b03-9ab0-690b651231ab


Forcing manual parse...
DEBUG: Successfully mapped headers: ['SID', 'AGE', 'GENDER', 'BMI', 'OAHI', 'AHI', 'MEAN_SAO2', 'AROUSAL INDEX', 'MEDICAL_HISTORY', 'SLEEP_DISORDERS']
✅ Success! 5 subjects ready.

--- Step 2: Applying Task ---
✅ Total sleep epochs created: 3984
Sample 0 Feature Length: 1920 (Expected: 1920)
Sample 0 Label: 0
